## **Rahven's VENTURI**

### Notebook 05 -- Building the FastF1 ML Base Table

##### **Objective**

This notebook implements the integration strategy established in
`03_prepare_fastf1.ipynb`.

It constructs the machine-learning-ready base table at the grain:

> **One row = One Driver × One Lap**

This notebook ONLY performs:

- Loading and merging `laps`, `results`, and `weather` (session-level datasets)
- Aggregating `car_data` and `position_data` (high-frequency telemetry) into
  lap-level features
- Producing one clean, versioned lap-level table for downstream feature
  engineering

It does NOT perform:

- Feature engineering for specific ML targets
- Target/label creation (pit stop, tyre degradation, race outcome, etc.)
- Model training

The four downstream models (lap time, tyre degradation, pit stop strategy,
race outcome) will all be built on top of the table produced here.

Author: Abdur Rahman - RAHVEN

Project: F1-AI / VENTURI

### Approach

We will NOT try to build the full 857-session base table in one shot.

We will:
1. Build and verify the pipeline on **one representative session** first
   (same session `03` used: 2018 Abu Dhabi GP, Practice 1).
2. Once the merge logic, row counts, and feature outputs are verified as
   correct on that single session, we will wrap it into a function and
   run it across the archive.

This mirrors the discipline from the Jolpica notebook -- verify small,
then scale.

In [1]:
from pathlib import Path
from collections import defaultdict
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(r"C:\F1-AI")
RAW_FASTF1_DIR = PROJECT_ROOT / "data" / "raw" / "fastf1"

assert RAW_FASTF1_DIR.exists(), f"FastF1 directory not found:\n{RAW_FASTF1_DIR}"

print(f"FastF1 Root : {RAW_FASTF1_DIR}")

FastF1 Root : C:\F1-AI\data\raw\fastf1


In [2]:
# Representative session for pipeline verification
# (same one used in 03 — 2018 Abu Dhabi GP, Practice 1)

SESSION_DIR = RAW_FASTF1_DIR / "2018" / "Abu_Dhabi_Grand_Prix" / "Practice_1"

assert SESSION_DIR.exists(), f"Session folder not found:\n{SESSION_DIR}"

print("Verification session:", SESSION_DIR)
print("Contents:", [p.name for p in SESSION_DIR.iterdir()])

Verification session: C:\F1-AI\data\raw\fastf1\2018\Abu_Dhabi_Grand_Prix\Practice_1
Contents: ['car_data', 'download_log.json', 'laps.parquet', 'metadata.json', 'position_data', 'results.parquet', 'weather.parquet']


In [3]:
laps_df = pd.read_parquet(SESSION_DIR / "laps.parquet")

print("Shape:", laps_df.shape)
print()
pd.set_option("display.max_rows", None)
print(laps_df.dtypes)

Shape: (460, 31)

Time                  timedelta64[ns]
Driver                         object
DriverNumber                   object
LapTime               timedelta64[ns]
LapNumber                     float64
Stint                         float64
PitOutTime            timedelta64[ns]
PitInTime             timedelta64[ns]
Sector1Time           timedelta64[ns]
Sector2Time           timedelta64[ns]
Sector3Time           timedelta64[ns]
Sector1SessionTime    timedelta64[ns]
Sector2SessionTime    timedelta64[ns]
Sector3SessionTime    timedelta64[ns]
SpeedI1                       float64
SpeedI2                       float64
SpeedFL                       float64
SpeedST                       float64
IsPersonalBest                   bool
Compound                       object
TyreLife                      float64
FreshTyre                        bool
Team                           object
LapStartTime          timedelta64[ns]
LapStartDate           datetime64[ns]
TrackStatus                    o

>**Identity / join keys**

Driver, DriverNumber, Team - driver and team identity for this lap. DriverNumber + LapNumber is our validated key from 03.

>**Core timing**

Time, LapTime, LapStartTime, LapStartDate — Time is the session-time timestamp when the lap was logged (usually lap end); LapStartTime/LapStartDate mark when the lap began. LapTime is the actual lap duration - this is a strong candidate target for model 1 (lap time prediction), so treat it as sacred, never as a feature for anything else.

>**Sector breakdown**

Sector1Time/2Time/3Time and their ...SessionTime counterparts - sector durations plus when each sector was completed in session-time. These sum to LapTime, so if LapTime is ever the target, sector times are leakage for that model (they encode the answer). They're fine as features for other targets like tyre degradation, though.

>**Speed traps**

SpeedI1, SpeedI2 (intermediate 1 & 2), SpeedFL (finish line), SpeedST (speed trap/straight) , this is exactly the telemetry signal I flagged earlier. These four numbers already summarize peak speed at four track points per lap, which may cover a big chunk of what we'd otherwise aggregate from raw car_data. Good news for scope.

>**Tyre state**

Compound, TyreLife, FreshTyre, Stint - compound (SOFT/MEDIUM/HARD/etc.), laps on current tyre, whether it's a never-used tyre, and stint number. This is the core input block for the tyre degradation model.

>**Pit info**

PitInTime, PitOutTime , non-null only on the lap a pit happened. This is literally our future label source for the pit-stop-strategy model (PitInTime.notna() → pitted this lap).

>**Session/track context**

TrackStatus, Position - TrackStatus is FastF1's coded string for flags (green/yellow/safety car/VSC/red flag etc.) and Position is track position at that point in the lap. Need to check the actual coding before trusting it.

>**Data-quality flags**

IsPersonalBest, Deleted, DeletedReason, IsAccurate, FastF1Generated - these tell us whether a lap's time was deleted by stewards, whether FastF1 synthetically reconstructed the lap (vs. measured it), and whether FastF1 itself trusts the timing as accurate. These matter a lot: a Deleted lap or a low-confidence lap could poison a lap-time regression target if we don't filter or flag it

In [4]:
inspect_cols = [
    "Compound", "TrackStatus", "FreshTyre",
    "Deleted", "DeletedReason", "IsAccurate", "FastF1Generated"]
for col in inspect_cols:
    print(f"--- {col} ---")
    print(laps_df[col].value_counts(dropna=False))
    print()

--- Compound ---
Compound
HYPERSOFT    289
ULTRASOFT    123
SUPERSOFT     48
Name: count, dtype: int64

--- TrackStatus ---
TrackStatus
1     399
12     57
21      4
Name: count, dtype: int64

--- FreshTyre ---
FreshTyre
False    281
True     179
Name: count, dtype: int64

--- Deleted ---
Deleted
False    460
Name: count, dtype: int64

--- DeletedReason ---
DeletedReason
    460
Name: count, dtype: int64

--- IsAccurate ---
IsAccurate
True     238
False    222
Name: count, dtype: int64

--- FastF1Generated ---
FastF1Generated
False    460
Name: count, dtype: int64



`TrackStatus` isn't one flag per lap - it's a sequence. Per FastF1's own docs: "a string that contains track status numbers for all track status that occurred during this lap." So "12" doesn't mean status-code-twelve — it means both status 1 and status 2 occurred during that lap (track was clear, then a yellow flag came out, mid-lap). "21" means the reverse order. This has a real consequence: we can't treat TrackStatus as a simple categorical column later - it needs to be parsed as a set/sequence of codes.

`IsAccurate` is not "lap time is correct." Docs are explicit: "Do not confuse this with the accuracy of the lap time or sector times." It only means lap start/end sync with neighboring laps.

`Compound` naming is NOT standardized across eras - this is a real cross-season problem for us. FastF1's docs say compound should normalize to SOFT/MEDIUM/HARD/INTERMEDIATE/WET (underlying C1–C5 compounds hidden). But our actual 2018 data shows HYPERSOFT/ULTRASOFT/SUPERSOFT — the old 2018-era 7-tier Pirelli naming, un-normalized. Since our archive spans 2018–2025, and F1 changed its tyre-naming scheme after 2018, any model trained across seasons will see inconsistent compound labels unless we build a mapping table ourselves (e.g., ordinal "softness rank" instead of raw string). Good catch to flag now before it becomes a silent bug in 05.

In [5]:
# Does IsAccurate correlate with box laps? (in/out laps are expected to be "inaccurate")

In [6]:

laps_df["is_box_lap"] = laps_df["PitInTime"].notna() | laps_df["PitOutTime"].notna()

print("IsAccurate vs is_box_lap:")
print(laps_df.groupby("is_box_lap")["IsAccurate"].value_counts())
print()

from collections import Counter
status_char_counts = Counter()
for status_string in laps_df["TrackStatus"]:
    for ch in str(status_string):
        status_char_counts[ch] += 1

print("Individual track status codes seen (any position in string):")
print(status_char_counts)

IsAccurate vs is_box_lap:
is_box_lap  IsAccurate
False       True          238
            False          52
True        False         170
Name: count, dtype: int64

Individual track status codes seen (any position in string):
Counter({'1': 460, '2': 61})


In [7]:
results_df = pd.read_parquet(SESSION_DIR / "results.parquet")

print("Shape:", results_df.shape)
print()
pd.set_option("display.max_rows", None)
print(results_df.dtypes)

Shape: (20, 22)

DriverNumber                   object
BroadcastName                  object
Abbreviation                   object
DriverId                       object
TeamName                       object
TeamColor                      object
TeamId                         object
FirstName                      object
LastName                       object
FullName                       object
HeadshotUrl                    object
CountryCode                    object
Position                      float64
ClassifiedPosition             object
GridPosition                  float64
Q1                    timedelta64[ns]
Q2                    timedelta64[ns]
Q3                    timedelta64[ns]
Time                  timedelta64[ns]
Status                         object
Points                        float64
Laps                          float64
dtype: object


In [8]:
#checking wether laps and result table have same columns?
laps_cols = set(laps_df.columns)
results_cols = set(results_df.columns)

overlap = laps_cols.intersection(results_cols)

print("Columns present in BOTH laps and results:")
print(overlap)

Columns present in BOTH laps and results:
{'Position', 'Time', 'DriverNumber'}


**Merging Check**
___

In [9]:
results_renamed = results_df.rename(columns={
    "Position": "FinalPosition",
    "Time": "ResultTime"
})

print(results_renamed.columns.tolist())

['DriverNumber', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName', 'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName', 'HeadshotUrl', 'CountryCode', 'FinalPosition', 'ClassifiedPosition', 'GridPosition', 'Q1', 'Q2', 'Q3', 'ResultTime', 'Status', 'Points', 'Laps']


In [10]:
before_rows = len(laps_df)

merged_df = laps_df.merge(
    results_renamed,
    on="DriverNumber",
    how="left"
)

after_rows = len(merged_df)

print("Rows before merge:", before_rows)
print("Rows after merge:", after_rows)
print("Row count matches:", before_rows == after_rows)

Rows before merge: 460
Rows after merge: 460
Row count matches: True


In [11]:
print("Any laps that failed to find a matching result row:")
print(merged_df["TeamName"].isna().sum())

Any laps that failed to find a matching result row:
0


In [12]:
mismatch = merged_df[merged_df["Team"] != merged_df["TeamName"]]

print("Rows where Team and TeamName disagree:", len(mismatch))
print()
print(mismatch[["Driver", "Team", "TeamName"]].drop_duplicates())

Rows where Team and TeamName disagree: 0

Empty DataFrame
Columns: [Driver, Team, TeamName]
Index: []


**Moving To Weather**
___

In [13]:
weather_df = pd.read_parquet(SESSION_DIR / "weather.parquet")

print("Shape:", weather_df.shape)
print()
print(weather_df.dtypes)

Shape: (111, 8)

Time             timedelta64[ns]
AirTemp                  float64
Humidity                 float64
Pressure                 float64
Rainfall                    bool
TrackTemp                float64
WindDirection              int64
WindSpeed                float64
dtype: object


In [14]:
duplicate_times = weather_df["Time"].duplicated().sum()
print("Duplicate Time values in this session's weather:", duplicate_times)

Duplicate Time values in this session's weather: 0


In [15]:
laps_sorted = merged_df.sort_values("LapStartTime").reset_index(drop=True)
weather_sorted = weather_df.sort_values("Time").reset_index(drop=True)

print(laps_sorted[["LapStartTime"]].head())
print()
print(weather_sorted[["Time"]].head())

            LapStartTime
0 0 days 00:14:32.810000
1 0 days 00:14:36.454000
2 0 days 00:14:58.922000
3 0 days 00:15:04.381000
4 0 days 00:15:09.463000

                    Time
0 0 days 00:00:57.228000
1 0 days 00:01:57.243000
2 0 days 00:02:57.257000
3 0 days 00:03:57.272000
4 0 days 00:04:57.283000


In [16]:
merged_with_weather = pd.merge_asof(
    laps_sorted,
    weather_sorted,
    left_on="LapStartTime",
    right_on="Time",
    direction="backward",
    suffixes=("", "_weather")
)

print("Rows before:", len(laps_sorted))
print("Rows after:", len(merged_with_weather))

Rows before: 460
Rows after: 460


In [17]:
missing_weather = merged_with_weather["AirTemp"].isna().sum()
print("Laps with no matching weather reading:", missing_weather)

Laps with no matching weather reading: 0


In [18]:
print([c for c in merged_with_weather.columns if "Time" in c])

['Time', 'LapTime', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'LapStartTime', 'ResultTime', 'Time_weather']


In [19]:
merged_with_weather["weather_lag_seconds"] = (
    merged_with_weather["LapStartTime"] - merged_with_weather["Time_weather"]
).dt.total_seconds()

print(merged_with_weather["weather_lag_seconds"].describe())

count    460.000000
mean      28.855326
std       17.161929
min        0.188000
25%       14.375750
50%       28.387000
75%       42.399250
max       59.882000
Name: weather_lag_seconds, dtype: float64


In [20]:
session_base = merged_with_weather.drop(columns=["Time_weather", "weather_lag_seconds"])

print("Base table shape:", session_base.shape)
print(session_base.columns.tolist())

Base table shape: (460, 60)
['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest', 'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime', 'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason', 'FastF1Generated', 'IsAccurate', 'is_box_lap', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamName', 'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName', 'HeadshotUrl', 'CountryCode', 'FinalPosition', 'ClassifiedPosition', 'GridPosition', 'Q1', 'Q2', 'Q3', 'ResultTime', 'Status', 'Points', 'Laps', 'AirTemp', 'Humidity', 'Pressure', 'Rainfall', 'TrackTemp', 'WindDirection', 'WindSpeed']


**its already 60 columns... !**

**Car data merging**
___

In [21]:
car_data_dir = SESSION_DIR / "car_data"

print("Driver files available:")
print(sorted(p.name for p in car_data_dir.iterdir()))

Driver files available:
['ALO.parquet', 'BOT.parquet', 'ERI.parquet', 'GAS.parquet', 'GIO.parquet', 'GRO.parquet', 'HAM.parquet', 'HAR.parquet', 'HUL.parquet', 'KUB.parquet', 'MAG.parquet', 'OCO.parquet', 'PER.parquet', 'RAI.parquet', 'RIC.parquet', 'SAI.parquet', 'STR.parquet', 'VAN.parquet', 'VER.parquet', 'VET.parquet']


In [22]:
sample_driver_file = car_data_dir / "HAM.parquet"

ham_car_data = pd.read_parquet(sample_driver_file)

print("Shape:", ham_car_data.shape)
print()
print(ham_car_data.dtypes)

Shape: (27177, 10)

Date            datetime64[ns]
RPM                    float64
Speed                  float64
nGear                    int64
Throttle               float64
Brake                     bool
DRS                      int64
Source                  object
Time           timedelta64[ns]
SessionTime    timedelta64[ns]
dtype: object


In [23]:
print(ham_car_data.head(10))

                     Date     RPM  Speed  nGear  Throttle  Brake  DRS Source  \
0 2018-11-23 08:23:16.092     0.0    0.0      0     104.0   True    8    car   
1 2018-11-23 08:46:52.867  5033.0    0.0      1       0.0  False    8    car   
2 2018-11-23 08:46:53.107  5039.0    0.0      1       0.0  False    8    car   
3 2018-11-23 08:46:53.347  5020.0    0.0      1       0.0  False    8    car   
4 2018-11-23 08:46:53.587  5026.0    0.0      1       0.0  False    8    car   
5 2018-11-23 08:46:53.827  5009.0    0.0      1       0.0  False    8    car   
6 2018-11-23 08:46:54.067  5004.0    0.0      1       0.0  False    8    car   
7 2018-11-23 08:46:54.268  4987.0    0.0      1       0.0  False    8    car   
8 2018-11-23 08:46:54.508  4989.0    0.0      1       0.0  False    8    car   
9 2018-11-23 08:46:54.748  4976.0    0.0      1       0.0  False    8    car   

                      Time              SessionTime  
0 -1 days +23:37:40.774000 -1 days +23:37:40.774000  
1   0 days 

In [24]:
print("DRS value counts:")
print(ham_car_data["DRS"].value_counts())
print()
print("Source value counts:")
print(ham_car_data["Source"].value_counts())

DRS value counts:
DRS
8     26446
12      680
14       29
10       22
Name: count, dtype: int64

Source value counts:
Source
car    27177
Name: count, dtype: int64


In [25]:
print("Time range covered by this driver's car_data:")
print("Min Time:", ham_car_data["Time"].min())
print("Max Time:", ham_car_data["Time"].max())
print()

ham_laps = session_base[session_base["Driver"] == "HAM"]
print("Time range covered by this driver's laps:")
print("Min LapStartTime:", ham_laps["LapStartTime"].min())
print("Max Time (lap end):", ham_laps["Time"].max())

Time range covered by this driver's car_data:
Min Time: -1 days +23:37:40.774000
Max Time: 0 days 01:51:36.900000

Time range covered by this driver's laps:
Min LapStartTime: 0 days 00:17:18.172000
Max Time (lap end): 0 days 01:50:00.635000


**Assigning each telemetry row to the correct lap**

In [26]:
ham_laps_sorted = ham_laps.sort_values("LapStartTime").reset_index(drop=True)

print(ham_laps_sorted[["LapNumber", "LapStartTime", "Time"]].head(10))

   LapNumber           LapStartTime                   Time
0        1.0 0 days 00:17:18.172000 0 days 00:19:26.421000
1        2.0 0 days 00:19:26.421000 0 days 00:35:57.544000
2        3.0 0 days 00:35:57.544000 0 days 00:38:33.169000
3        4.0 0 days 00:38:33.169000 0 days 00:40:12.844000
4        5.0 0 days 00:40:12.844000 0 days 00:43:11.643000
5        6.0 0 days 00:43:11.643000 0 days 00:44:51.219000
6        7.0 0 days 00:44:51.219000 0 days 00:47:45.414000
7        8.0 0 days 00:47:45.414000 0 days 00:49:35.170000
8        9.0 0 days 00:49:35.170000 0 days 01:07:19.809000
9       10.0 0 days 01:07:19.809000 0 days 01:08:59.352000


!!! lap 2 and lap 9 are unusually long, over 15 minutes each, that's probably a red flag stoppage since we know this session had 4 laps under status code 21.

In [27]:
ham_car_sorted = ham_car_data.sort_values("Time").reset_index(drop=True)

lap_boundaries = ham_laps_sorted[["LapNumber", "LapStartTime"]].sort_values("LapStartTime")

ham_car_with_lap = pd.merge_asof(
    ham_car_sorted,
    lap_boundaries,
    left_on="Time",
    right_on="LapStartTime",
    direction="backward"
)

print(ham_car_with_lap["LapNumber"].value_counts(dropna=False).sort_index())

LapNumber
1.0      525
2.0     4068
3.0      639
4.0      409
5.0      737
6.0      408
7.0      716
8.0      453
9.0     4364
10.0     408
11.0     662
12.0     636
13.0     483
14.0     437
15.0     618
16.0    2288
17.0     432
18.0     432
19.0     431
20.0     430
21.0     433
22.0     436
23.0     448
24.0     662
25.0     563
26.0    1116
NaN     3943
Name: count, dtype: int64


Normal laps sit in the 400 to 700 row range, roughly matching a 4Hz-ish sampling rate over a 1.5 to 2 minute lap. Laps 2 and 9 blow up to over 4000 rows each, matching what we already suspected, those were the red flag laps that lasted 15+ minutes. Lap 16 also looks unusually large at 2288, worth a quick check since we didn't flag that one earlier.

In [28]:
lap16_check = ham_laps_sorted[ham_laps_sorted["LapNumber"] == 16.0]
print(lap16_check[["LapNumber", "LapStartTime", "Time", "TrackStatus", "PitInTime", "PitOutTime"]])

    LapNumber           LapStartTime                   Time TrackStatus  \
15       16.0 0 days 01:20:29.283000 0 days 01:29:46.809000           1   

   PitInTime             PitOutTime  
15       NaT 0 days 01:27:56.517000  


... Confirmed, **NO ISSUE** . Lap 16 has a PitOutTime, meaning this is the out lap right after a pit stop, car comes out of the pits and then completes the lap. Pit lane traversal plus getting back up to speed naturally makes it longer and captures more telemetry. Nothing wrong with the data.

In [29]:
ham_car_with_lap_clean = ham_car_with_lap.dropna(subset=["LapNumber"])
lap_telemetry_features = ham_car_with_lap_clean.groupby("LapNumber").agg(
    avg_speed=("Speed", "mean"),
    max_speed=("Speed", "max"),
    avg_throttle=("Throttle", "mean"),
    full_throttle_pct=("Throttle", lambda x: (x == 100).mean() * 100),
    max_rpm=("RPM", "max"),
    brake_sample_pct=("Brake", "mean"),
    drs_active_pct=("DRS", lambda x: x.isin([10, 12, 14]).mean() * 100),
    telemetry_sample_count=("Speed", "count")
).reset_index()

print(lap_telemetry_features.head(10))

   LapNumber   avg_speed  max_speed  avg_throttle  full_throttle_pct  max_rpm  \
0        1.0  145.516190      263.0     47.784762          22.857143  12804.0   
1        2.0   20.359636      301.0     86.682399           1.474926  12664.0   
2        3.0  127.517997      282.0     32.361502           4.851330  12446.0   
3        4.0  199.112469      316.0     68.249389          53.789731  12479.0   
4        5.0  111.358209      267.0     22.187246           2.985075  12388.0   
5        6.0  199.504902      316.0     68.941176          54.411765  12766.0   
6        7.0  114.174581      270.0     24.642458           7.541899  12759.0   
7        8.0  179.746137      316.0     53.549669          38.189845  12758.0   
8        9.0   18.780477      288.0     86.830431           2.176902  12441.0   
9       10.0  199.401961      324.0     66.811275          49.754902  12960.0   

   brake_sample_pct  drs_active_pct  telemetry_sample_count  
0          0.230476        0.000000           

In [30]:
lap2_telemetry = ham_car_with_lap_clean[ham_car_with_lap_clean["LapNumber"] == 2.0]

print(lap2_telemetry["Throttle"].value_counts().sort_index(ascending=False).head(15))

Throttle
104.0    3165
100.0      60
99.0        2
98.0        1
96.0        1
92.0        1
91.0        3
90.0        1
89.0        1
88.0        2
83.0        2
82.0        1
80.0        1
79.0        1
78.0        3
Name: count, dtype: int64


>FastF1 docs mentioned something specific about this, a throttle value of 104 is a known error code that shows up specifically when a car is stationary in the pits or on the grid.

>3165 out of 4068 rows in this one lap alone are the 104 error code, that is a massive contamination, not just a few stray points. This would badly distort our throttle features specifically for any lap involving stopped time, safety car, red flag, or even just sitting in the pit box.

In [31]:
ham_car_with_lap_clean = ham_car_with_lap_clean.copy()
ham_car_with_lap_clean["Throttle_clean"] = ham_car_with_lap_clean["Throttle"].replace(104.0, np.nan)

lap_telemetry_features = ham_car_with_lap_clean.groupby("LapNumber").agg(
    avg_speed=("Speed", "mean"),
    max_speed=("Speed", "max"),
    avg_throttle=("Throttle_clean", "mean"),
    full_throttle_pct=("Throttle_clean", lambda x: (x == 100).mean() * 100),
    max_rpm=("RPM", "max"),
    brake_sample_pct=("Brake", "mean"),
    drs_active_pct=("DRS", lambda x: x.isin([10, 12, 14]).mean() * 100),
    telemetry_sample_count=("Speed", "count")
).reset_index()

print(lap_telemetry_features.head(10))

   LapNumber   avg_speed  max_speed  avg_throttle  full_throttle_pct  max_rpm  \
0        1.0  145.516190      263.0     47.784762          22.857143  12804.0   
1        2.0   20.359636      301.0     25.984496           1.474926  12664.0   
2        3.0  127.517997      282.0     32.361502           4.851330  12446.0   
3        4.0  199.112469      316.0     68.249389          53.789731  12479.0   
4        5.0  111.358209      267.0     22.187246           2.985075  12388.0   
5        6.0  199.504902      316.0     68.941176          54.411765  12766.0   
6        7.0  114.174581      270.0     24.642458           7.541899  12759.0   
7        8.0  179.746137      316.0     53.549669          38.189845  12758.0   
8        9.0   18.780477      288.0     25.294118           2.176902  12441.0   
9       10.0  199.401961      324.0     66.811275          49.754902  12960.0   

   brake_sample_pct  drs_active_pct  telemetry_sample_count  
0          0.230476        0.000000           

**Generalize it into a function and run it across all 20 drivers in this session.**
___

In [32]:
driver_code_to_number = session_base[["Driver", "DriverNumber"]].drop_duplicates().set_index("Driver")["DriverNumber"].to_dict()

print(driver_code_to_number)

{'RAI': '7', 'ERI': '9', 'VAN': '2', 'KUB': '40', 'SAI': '55', 'GRO': '8', 'HAR': '28', 'GIO': '36', 'STR': '18', 'GAS': '10', 'VET': '5', 'BOT': '77', 'VER': '33', 'ALO': '14', 'MAG': '20', 'PER': '11', 'HUL': '27', 'RIC': '3', 'OCO': '31', 'HAM': '44'}


In [33]:
def build_lap_telemetry_features(car_df, lap_boundaries_df):
    car_sorted = car_df.sort_values("Time").reset_index(drop=True)
    boundaries_sorted = lap_boundaries_df.sort_values("LapStartTime").reset_index(drop=True)

    car_with_lap = pd.merge_asof(
        car_sorted,
        boundaries_sorted,
        left_on="Time",
        right_on="LapStartTime",
        direction="backward"
    )

    car_with_lap = car_with_lap.dropna(subset=["LapNumber"]).copy()
    car_with_lap["Throttle_clean"] = car_with_lap["Throttle"].replace(104.0, np.nan)

    features = car_with_lap.groupby("LapNumber").agg(
        avg_speed=("Speed", "mean"),
        max_speed=("Speed", "max"),
        avg_throttle=("Throttle_clean", "mean"),
        full_throttle_pct=("Throttle_clean", lambda x: (x == 100).mean() * 100),
        max_rpm=("RPM", "max"),
        brake_sample_pct=("Brake", "mean"),
        drs_active_pct=("DRS", lambda x: x.isin([10, 12, 14]).mean() * 100),
        telemetry_sample_count=("Speed", "count")
    ).reset_index()

    return features

In [34]:
all_driver_telemetry = []

for driver_code, driver_number in driver_code_to_number.items():
    car_file = car_data_dir / f"{driver_code}.parquet"

    if not car_file.exists():
        print(f"Missing car_data file for {driver_code}, skipping")
        continue

    driver_car_df = pd.read_parquet(car_file)

    driver_lap_boundaries = session_base[session_base["DriverNumber"] == driver_number][["LapNumber", "LapStartTime"]]

    driver_features = build_lap_telemetry_features(driver_car_df, driver_lap_boundaries)
    driver_features["DriverNumber"] = driver_number

    all_driver_telemetry.append(driver_features)

session_telemetry_features = pd.concat(all_driver_telemetry, ignore_index=True)

print("Shape:", session_telemetry_features.shape)
print(session_telemetry_features.head())

Shape: (460, 10)
   LapNumber   avg_speed  max_speed  avg_throttle  full_throttle_pct  max_rpm  \
0        1.0  142.090573      241.0     51.772643          35.120148  12563.0   
1        2.0  117.994194      241.0     46.716981          31.349782  12247.0   
2        3.0  121.811012      241.0     50.395833          34.821429  11888.0   
3        4.0   19.169044      305.0     31.969231           3.133037  12752.0   
4        5.0  196.980723      316.0     66.412048          47.228916  12534.0   

   brake_sample_pct  drs_active_pct  telemetry_sample_count DriverNumber  
0          0.181146        0.000000                     541            7  
1          0.152395        0.000000                     689            7  
2          0.194940        0.000000                     672            7  
3          0.838438        0.794950                    4277            7  
4          0.187952       17.349398                     415            7  


In [35]:
print("Total laps in session_base:", len(session_base))
print("Total laps with telemetry features:", len(session_telemetry_features))

Total laps in session_base: 460
Total laps with telemetry features: 460


In [36]:
before_rows = len(session_base)

session_with_telemetry = session_base.merge(
    session_telemetry_features,
    on=["DriverNumber", "LapNumber"],
    how="left"
)

after_rows = len(session_with_telemetry)

print("Rows before:", before_rows)
print("Rows after:", after_rows)
print("Match:", before_rows == after_rows)

Rows before: 460
Rows after: 460
Match: True


In [37]:
telemetry_cols = ["avg_speed", "max_speed", "avg_throttle", "full_throttle_pct", "max_rpm", "brake_sample_pct", "drs_active_pct"]

print("Missing values per telemetry column:")
print(session_with_telemetry[telemetry_cols].isna().sum())

Missing values per telemetry column:
avg_speed            0
max_speed            0
avg_throttle         0
full_throttle_pct    0
max_rpm              0
brake_sample_pct     0
drs_active_pct       0
dtype: int64


**Moving to Position Data**
___

In [38]:
position_data_dir = SESSION_DIR / "position_data"

print("Driver files available:")
print(sorted(p.name for p in position_data_dir.iterdir()))

Driver files available:
['ALO.parquet', 'BOT.parquet', 'ERI.parquet', 'GAS.parquet', 'GIO.parquet', 'GRO.parquet', 'HAM.parquet', 'HAR.parquet', 'HUL.parquet', 'KUB.parquet', 'MAG.parquet', 'OCO.parquet', 'PER.parquet', 'RAI.parquet', 'RIC.parquet', 'SAI.parquet', 'STR.parquet', 'VAN.parquet', 'VER.parquet', 'VET.parquet']


In [39]:
ham_position_data = pd.read_parquet(position_data_dir / "HAM.parquet")

print("Shape:", ham_position_data.shape)
print()
print(ham_position_data.dtypes)
print()
print(ham_position_data.head(10))

Shape: (22272, 8)

Date            datetime64[ns]
Status                  object
X                      float64
Y                      float64
Z                      float64
Source                  object
Time           timedelta64[ns]
SessionTime    timedelta64[ns]
dtype: object

                     Date   Status        X       Y    Z Source  \
0 2018-11-23 08:46:44.023  OnTrack -10797.0 -2558.0  0.0    pos   
1 2018-11-23 08:46:44.323  OnTrack -10797.0 -2558.0  0.0    pos   
2 2018-11-23 08:46:44.623  OnTrack -10797.0 -2558.0  0.0    pos   
3 2018-11-23 08:46:44.923  OnTrack -10797.0 -2558.0  0.0    pos   
4 2018-11-23 08:46:45.223  OnTrack -10797.0 -2558.0  0.0    pos   
5 2018-11-23 08:46:45.523  OnTrack -10797.0 -2558.0  0.0    pos   
6 2018-11-23 08:46:45.822  OnTrack -10797.0 -2558.0  0.0    pos   
7 2018-11-23 08:46:46.122  OnTrack -10797.0 -2558.0  0.0    pos   
8 2018-11-23 08:46:46.422  OnTrack -10797.0 -2558.0  0.0    pos   
9 2018-11-23 08:46:46.722  OnTrack -10797.0 -255

In [40]:
print("Status value counts across Hamilton's full session:")
print(ham_position_data["Status"].value_counts())

Status value counts across Hamilton's full session:
Status
OnTrack    22272
Name: count, dtype: int64


**!!! We are not going to overengineered this position_data into a feature set right now**

**Final Clean-up on the single session table before we scale this ro whole archive.**
____

In [41]:
columns_to_drop = [
    "TeamName",
    "SessionTime" if "SessionTime" in session_with_telemetry.columns else None,
    "is_box_lap",
    "telemetry_sample_count"
]
columns_to_drop = [c for c in columns_to_drop if c is not None]

session_final = session_with_telemetry.drop(columns=columns_to_drop)

print("Final single session base table shape:", session_final.shape)
print(session_final.columns.tolist())

Final single session base table shape: (460, 65)
['Time', 'Driver', 'DriverNumber', 'LapTime', 'LapNumber', 'Stint', 'PitOutTime', 'PitInTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Sector1SessionTime', 'Sector2SessionTime', 'Sector3SessionTime', 'SpeedI1', 'SpeedI2', 'SpeedFL', 'SpeedST', 'IsPersonalBest', 'Compound', 'TyreLife', 'FreshTyre', 'Team', 'LapStartTime', 'LapStartDate', 'TrackStatus', 'Position', 'Deleted', 'DeletedReason', 'FastF1Generated', 'IsAccurate', 'BroadcastName', 'Abbreviation', 'DriverId', 'TeamColor', 'TeamId', 'FirstName', 'LastName', 'FullName', 'HeadshotUrl', 'CountryCode', 'FinalPosition', 'ClassifiedPosition', 'GridPosition', 'Q1', 'Q2', 'Q3', 'ResultTime', 'Status', 'Points', 'Laps', 'AirTemp', 'Humidity', 'Pressure', 'Rainfall', 'TrackTemp', 'WindDirection', 'WindSpeed', 'avg_speed', 'max_speed', 'avg_throttle', 'full_throttle_pct', 'max_rpm', 'brake_sample_pct', 'drs_active_pct']


**The Function itself**

In [42]:
def parse_session_path(session_dir):
    session_name = session_dir.name
    event_name = session_dir.parent.name
    season = session_dir.parent.parent.name
    return season, event_name, session_name


def process_session(session_dir):
    season, event_name, session_name = parse_session_path(session_dir)

    laps_df = pd.read_parquet(session_dir / "laps.parquet")
    results_df = pd.read_parquet(session_dir / "results.parquet")
    weather_df = pd.read_parquet(session_dir / "weather.parquet")

    results_renamed = results_df.rename(columns={
        "Position": "FinalPosition",
        "Time": "ResultTime"
    })

    merged_df = laps_df.merge(results_renamed, on="DriverNumber", how="left")

    weather_df = weather_df.groupby("Time", as_index=False).mean(numeric_only=True)

    laps_sorted = merged_df.sort_values("LapStartTime").reset_index(drop=True)
    weather_sorted = weather_df.sort_values("Time").reset_index(drop=True)

    merged_with_weather = pd.merge_asof(
        laps_sorted,
        weather_sorted,
        left_on="LapStartTime",
        right_on="Time",
        direction="backward",
        suffixes=("", "_weather")
    )

    driver_code_to_number = merged_with_weather[["Driver", "DriverNumber"]].drop_duplicates().set_index("Driver")["DriverNumber"].to_dict()

    car_data_dir = session_dir / "car_data"
    all_driver_telemetry = []

    for driver_code, driver_number in driver_code_to_number.items():
        car_file = car_data_dir / f"{driver_code}.parquet"
        if not car_file.exists():
            continue

        driver_car_df = pd.read_parquet(car_file)
        driver_lap_boundaries = merged_with_weather[merged_with_weather["DriverNumber"] == driver_number][["LapNumber", "LapStartTime"]]

        driver_features = build_lap_telemetry_features(driver_car_df, driver_lap_boundaries)
        driver_features["DriverNumber"] = driver_number
        all_driver_telemetry.append(driver_features)

    if len(all_driver_telemetry) == 0:
        session_telemetry_features = pd.DataFrame(columns=["LapNumber", "DriverNumber"])
    else:
        session_telemetry_features = pd.concat(all_driver_telemetry, ignore_index=True)

    session_with_telemetry = merged_with_weather.merge(
        session_telemetry_features,
        on=["DriverNumber", "LapNumber"],
        how="left"
    )

    session_with_telemetry["Season"] = season
    session_with_telemetry["EventName"] = event_name
    session_with_telemetry["SessionName"] = session_name

    columns_to_drop = ["TeamName", "Time_weather"]
    columns_to_drop = [c for c in columns_to_drop if c in session_with_telemetry.columns]
    session_final = session_with_telemetry.drop(columns=columns_to_drop)

    return session_final

In [43]:
test_session_dir = RAW_FASTF1_DIR / "2018" / "Abu_Dhabi_Grand_Prix" / "Practice_1"

result = process_session(test_session_dir)

print("Shape:", result.shape)
print("Matches our manual version:", result.shape == session_final.shape)

Shape: (460, 69)
Matches our manual version: False


In [44]:
manual_cols = set(session_final.columns)
function_cols = set(result.columns)

print("In function output but NOT in manual version:")
print(function_cols - manual_cols)
print()
print("In manual version but NOT in function output:")
print(manual_cols - function_cols)

In function output but NOT in manual version:
{'telemetry_sample_count', 'EventName', 'SessionName', 'Season'}

In manual version but NOT in function output:
set()


In [45]:
def process_session(session_dir):
    season, event_name, session_name = parse_session_path(session_dir)

    laps_df = pd.read_parquet(session_dir / "laps.parquet")
    results_df = pd.read_parquet(session_dir / "results.parquet")
    weather_df = pd.read_parquet(session_dir / "weather.parquet")

    results_renamed = results_df.rename(columns={
        "Position": "FinalPosition",
        "Time": "ResultTime"
    })

    merged_df = laps_df.merge(results_renamed, on="DriverNumber", how="left")

    weather_df = weather_df.groupby("Time", as_index=False).mean(numeric_only=True)

    laps_sorted = merged_df.sort_values("LapStartTime").reset_index(drop=True)
    weather_sorted = weather_df.sort_values("Time").reset_index(drop=True)

    merged_with_weather = pd.merge_asof(
        laps_sorted,
        weather_sorted,
        left_on="LapStartTime",
        right_on="Time",
        direction="backward",
        suffixes=("", "_weather")
    )

    driver_code_to_number = merged_with_weather[["Driver", "DriverNumber"]].drop_duplicates().set_index("Driver")["DriverNumber"].to_dict()

    car_data_dir = session_dir / "car_data"
    all_driver_telemetry = []

    for driver_code, driver_number in driver_code_to_number.items():
        car_file = car_data_dir / f"{driver_code}.parquet"
        if not car_file.exists():
            continue

        driver_car_df = pd.read_parquet(car_file)
        driver_lap_boundaries = merged_with_weather[merged_with_weather["DriverNumber"] == driver_number][["LapNumber", "LapStartTime"]]

        driver_features = build_lap_telemetry_features(driver_car_df, driver_lap_boundaries)
        driver_features["DriverNumber"] = driver_number
        all_driver_telemetry.append(driver_features)

    if len(all_driver_telemetry) == 0:
        session_telemetry_features = pd.DataFrame(columns=["LapNumber", "DriverNumber"])
    else:
        session_telemetry_features = pd.concat(all_driver_telemetry, ignore_index=True)

    session_with_telemetry = merged_with_weather.merge(
        session_telemetry_features,
        on=["DriverNumber", "LapNumber"],
        how="left"
    )

    session_with_telemetry["Season"] = season
    session_with_telemetry["EventName"] = event_name
    session_with_telemetry["SessionName"] = session_name

    columns_to_drop = ["TeamName", "Time_weather", "telemetry_sample_count"]
    columns_to_drop = [c for c in columns_to_drop if c in session_with_telemetry.columns]
    session_final = session_with_telemetry.drop(columns=columns_to_drop)

    return session_final

In [46]:
result = process_session(test_session_dir)

print("Shape:", result.shape)
print("Extra cols now:", set(result.columns) - set(session_final.columns))

Shape: (460, 68)
Extra cols now: {'EventName', 'SessionName', 'Season'}


### **Function verified !!!!**

**But before implenting this function for all sessions, lets check this function manually for 2-3 more sessions - in case that function hits any edge case.**

In [47]:
sample_metadata_path = test_session_dir / "metadata.json"

import json
with open(sample_metadata_path) as f:
    sample_metadata = json.load(f)

print(sample_metadata)

{'season': 2018, 'round': 21, 'event': 'Abu Dhabi Grand Prix', 'official_name': 'FORMULA 1 2018 ETIHAD AIRWAYS ABU DHABI GRAND PRIX', 'country': 'United Arab Emirates', 'location': 'Yas Marina', 'session': 'Practice 1', 'event_format': 'conventional', 'f1_api_support': True, 'drivers': 20, 'total_laps': None, 'download_time': '2026-07-20T03:06:01.714163', 'fastf1_version': '3.8.3'}


In [48]:
catalog_rows = []

for metadata_file in RAW_FASTF1_DIR.rglob("metadata.json"):
    with open(metadata_file, encoding="utf-8") as f:
        meta = json.load(f)
    meta["folder"] = str(metadata_file.parent)
    catalog_rows.append(meta)

catalog_df = pd.DataFrame(catalog_rows)

print("Total sessions catalogued:", len(catalog_df))
print()
print("Session types:")
print(catalog_df["session"].value_counts())
print()
print("Event formats:")
print(catalog_df["event_format"].value_counts())

Total sessions catalogued: 857

Session types:
session
Practice 1           173
Qualifying           173
Race                 173
Practice 2           154
Practice 3           148
Sprint                24
Sprint Qualifying     12
Name: count, dtype: int64

Event formats:
event_format
conventional         743
sprint_qualifying     60
sprint                30
sprint_shootout       24
Name: count, dtype: int64


In [49]:
qualifying_pick = catalog_df[catalog_df["session"] == "Qualifying"].iloc[0]
sprint_pick = catalog_df[catalog_df["session"] == "Sprint"].iloc[0]

belgium_2021_race = catalog_df[
    (catalog_df["season"] == 2021) &
    (catalog_df["event"].str.contains("Belg", case=False, na=False)) &
    (catalog_df["session"] == "Race")
]

print("Qualifying pick:", qualifying_pick["folder"])
print()
print("Sprint pick:", sprint_pick["folder"])
print()
print("Belgium 2021 Race match:")
print(belgium_2021_race[["event", "session", "folder"]])

Qualifying pick: C:\F1-AI\data\raw\fastf1\2025\United_States_Grand_Prix\Qualifying

Sprint pick: C:\F1-AI\data\raw\fastf1\2025\United_States_Grand_Prix\Sprint

Belgium 2021 Race match:
                  event session  \
543  Belgian Grand Prix    Race   

                                                folder  
543  C:\F1-AI\data\raw\fastf1\2021\Belgian_Grand_Pr...  


In [50]:
edge_case_sessions = {
    "Qualifying": Path(qualifying_pick["folder"]),
    "Sprint": Path(sprint_pick["folder"]),
    "Belgium 2021 Race": Path(belgium_2021_race.iloc[0]["folder"])
}

edge_case_results = {}

for label, path in edge_case_sessions.items():
    print(f"Testing: {label}")
    try:
        out = process_session(path)
        edge_case_results[label] = out
        print(f"  Success, shape: {out.shape}")
    except Exception as e:
        print(f"  FAILED: {type(e).__name__}: {e}")
    print()

Testing: Qualifying
  Success, shape: (285, 68)

Testing: Sprint
  Success, shape: (320, 68)

Testing: Belgium 2021 Race
  Success, shape: (60, 68)



In [51]:
belgium_result = edge_case_results["Belgium 2021 Race"]

telemetry_cols = ["avg_speed", "max_speed", "avg_throttle", "full_throttle_pct", "max_rpm", "brake_sample_pct", "drs_active_pct"]

print("Missing telemetry values in Belgium 2021:")
print(belgium_result[telemetry_cols].isna().sum())
print()
print("Laps per driver:")
print(belgium_result["LapNumber"].value_counts().sort_index())

Missing telemetry values in Belgium 2021:
avg_speed            0
max_speed            0
avg_throttle         0
full_throttle_pct    0
max_rpm              0
brake_sample_pct     0
drs_active_pct       0
dtype: int64

Laps per driver:
LapNumber
1.0    20
2.0    20
3.0    20
Name: count, dtype: int64


In [52]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "fastf1_lap_base"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Output directory:", PROCESSED_DIR)

Output directory: C:\F1-AI\data\processed\fastf1_lap_base


In [53]:
def get_output_path(season, event_name, session_name):
    safe_event = event_name.replace(" ", "_")
    safe_session = session_name.replace(" ", "_")
    filename = f"{season}_{safe_event}_{safe_session}.parquet"
    return PROCESSED_DIR / filename

In [54]:
all_session_folders = [Path(f) for f in catalog_df["folder"]]

print("Total sessions to process:", len(all_session_folders))

Total sessions to process: 857


In [55]:
from tqdm import tqdm

In [56]:
success_count = 0
skipped_count = 0
failed_sessions = []

for _, row in tqdm(catalog_df.iterrows(), total=len(catalog_df), desc="Processing sessions"):
    season = row["season"]
    event_name = row["event"]
    session_name = row["session"]
    folder = Path(row["folder"])

    output_path = get_output_path(season, event_name, session_name)

    if output_path.exists():
        skipped_count += 1
        continue

    try:
        session_df = process_session(folder)
        session_df.to_parquet(output_path, index=False)
        success_count += 1
    except Exception as e:
        failed_sessions.append({
            "folder": str(folder),
            "error_type": type(e).__name__,
            "error_message": str(e)
        })

print()
print("Done.")
print("Newly processed:", success_count)
print("Already existed, skipped:", skipped_count)
print("Failed:", len(failed_sessions))

Processing sessions: 100%|██████████████████████████████████████████████████████████| 857/857 [00:00<00:00, 920.57it/s]


Done.
Newly processed: 0
Already existed, skipped: 850
Failed: 7


In [57]:
failed_df = pd.DataFrame(failed_sessions)

print("Total failures:", len(failed_df))
print()
print("Error types breakdown:")
print(failed_df["error_type"].value_counts())

Total failures: 7

Error types breakdown:
error_type
FileNotFoundError    7
Name: count, dtype: int64


In [58]:
pd.set_option("display.max_colwidth", None)
print(failed_df[["folder", "error_type", "error_message"]])

                                                         folder  \
0   C:\F1-AI\data\raw\fastf1\2021\Russian_Grand_Prix\Practice_3   
1   C:\F1-AI\data\raw\fastf1\2020\Styrian_Grand_Prix\Practice_3   
2     C:\F1-AI\data\raw\fastf1\2020\Eifel_Grand_Prix\Practice_1   
3     C:\F1-AI\data\raw\fastf1\2020\Eifel_Grand_Prix\Practice_2   
4  C:\F1-AI\data\raw\fastf1\2019\Japanese_Grand_Prix\Practice_3   
5         C:\F1-AI\data\raw\fastf1\2018\Italian_Grand_Prix\Race   
6    C:\F1-AI\data\raw\fastf1\2018\German_Grand_Prix\Practice_1   

          error_type  \
0  FileNotFoundError   
1  FileNotFoundError   
2  FileNotFoundError   
3  FileNotFoundError   
4  FileNotFoundError   
5  FileNotFoundError   
6  FileNotFoundError   

                                                                                                              error_message  
0   [Errno 2] No such file or directory: 'C:\\F1-AI\\data\\raw\\fastf1\\2021\\Russian_Grand_Prix\\Practice_3\\laps.parquet'  
1   [Errno 2] No s

In [59]:
broken_folder = Path(r"C:\F1-AI\data\raw\fastf1\2024\Bahrain_Grand_Prix\Race")

print("Actual contents of this folder:")
print([p.name for p in broken_folder.iterdir()])

Actual contents of this folder:
['car_data', 'download_log.json', 'laps.parquet', 'metadata.json', 'position_data', 'results.parquet']


In [60]:
null_key_folder = Path(r"C:\F1-AI\data\raw\fastf1\2025\Spanish_Grand_Prix\Practice_3")

laps_check = pd.read_parquet(null_key_folder / "laps.parquet")

print("Total laps in this session:", len(laps_check))
print("Laps with null LapStartTime:", laps_check["LapStartTime"].isna().sum())
print()
print(laps_check[laps_check["LapStartTime"].isna()][["Driver", "LapNumber", "PitOutTime", "PitInTime", "Time"]])

Total laps in this session: 312
Laps with null LapStartTime: 19

    Driver  LapNumber PitOutTime PitInTime                     Time
0      NOR        1.0        NaT       NaT -1 days +23:57:45.195000
17     TSU        1.0        NaT       NaT          0 days 00:00:00
29     OCO        1.0        NaT       NaT   0 days 00:04:47.116000
41     BOR        1.0        NaT       NaT          0 days 00:00:00
55     GAS        1.0        NaT       NaT          0 days 00:00:00
74     COL        1.0        NaT       NaT -1 days +23:58:43.791000
92     HUL        1.0        NaT       NaT          0 days 00:00:00
109    STR        1.0        NaT       NaT -1 days +23:58:45.542000
128    ALO        1.0        NaT       NaT -1 days +23:58:45.759000
144    BEA        1.0        NaT       NaT -1 days +23:57:31.262000
156    SAI        1.0        NaT       NaT          0 days 00:00:00
172    LAW        1.0        NaT       NaT -1 days +23:58:45.353000
191    VER        1.0        NaT       NaT -1 days 

In [61]:
def process_session(session_dir):
    season, event_name, session_name = parse_session_path(session_dir)

    laps_df = pd.read_parquet(session_dir / "laps.parquet")
    results_df = pd.read_parquet(session_dir / "results.parquet")

    results_renamed = results_df.rename(columns={
        "Position": "FinalPosition",
        "Time": "ResultTime"
    })

    merged_df = laps_df.merge(results_renamed, on="DriverNumber", how="left")

    # Fix 1: drop laps with no recorded start time (mostly lap 1, structural FastF1 gap)
    merged_df = merged_df[merged_df["LapStartTime"].notna()].copy()

    laps_sorted = merged_df.sort_values("LapStartTime").reset_index(drop=True)

    # Fix 2: weather.parquet sometimes missing, merge only if present
    weather_path = session_dir / "weather.parquet"
    if weather_path.exists():
        weather_df = pd.read_parquet(weather_path)
        weather_df = weather_df.groupby("Time", as_index=False).mean(numeric_only=True)
        weather_sorted = weather_df.sort_values("Time").reset_index(drop=True)

        merged_with_weather = pd.merge_asof(
            laps_sorted,
            weather_sorted,
            left_on="LapStartTime",
            right_on="Time",
            direction="backward",
            suffixes=("", "_weather")
        )
    else:
        merged_with_weather = laps_sorted.copy()
        for col in ["AirTemp", "Humidity", "Pressure", "Rainfall", "TrackTemp", "WindDirection", "WindSpeed"]:
            merged_with_weather[col] = np.nan

    driver_code_to_number = merged_with_weather[["Driver", "DriverNumber"]].drop_duplicates().set_index("Driver")["DriverNumber"].to_dict()

    car_data_dir = session_dir / "car_data"
    all_driver_telemetry = []

    for driver_code, driver_number in driver_code_to_number.items():
        car_file = car_data_dir / f"{driver_code}.parquet"
        if not car_file.exists():
            continue

        driver_car_df = pd.read_parquet(car_file)
        driver_lap_boundaries = merged_with_weather[merged_with_weather["DriverNumber"] == driver_number][["LapNumber", "LapStartTime"]]

        driver_features = build_lap_telemetry_features(driver_car_df, driver_lap_boundaries)
        driver_features["DriverNumber"] = driver_number
        all_driver_telemetry.append(driver_features)

    if len(all_driver_telemetry) == 0:
        session_telemetry_features = pd.DataFrame(columns=["LapNumber", "DriverNumber"])
    else:
        session_telemetry_features = pd.concat(all_driver_telemetry, ignore_index=True)

    session_with_telemetry = merged_with_weather.merge(
        session_telemetry_features,
        on=["DriverNumber", "LapNumber"],
        how="left"
    )

    session_with_telemetry["Season"] = season
    session_with_telemetry["EventName"] = event_name
    session_with_telemetry["SessionName"] = session_name

    columns_to_drop = ["TeamName", "Time_weather", "telemetry_sample_count"]
    columns_to_drop = [c for c in columns_to_drop if c in session_with_telemetry.columns]
    session_final = session_with_telemetry.drop(columns=columns_to_drop)

    return session_final

In [62]:
retry_success = 0
retry_failed = []

for fail in tqdm(failed_sessions, desc="Retrying failed sessions"):
    folder = Path(fail["folder"])
    row = catalog_df[catalog_df["folder"] == str(folder)].iloc[0]

    output_path = get_output_path(row["season"], row["event"], row["session"])

    try:
        session_df = process_session(folder)
        session_df.to_parquet(output_path, index=False)
        retry_success += 1
    except Exception as e:
        retry_failed.append({
            "folder": str(folder),
            "error_type": type(e).__name__,
            "error_message": str(e)
        })

print("Retried:", len(failed_sessions))
print("Now succeeded:", retry_success)
print("Still failed:", len(retry_failed))
if retry_failed:
    print(retry_failed)

Retrying failed sessions: 100%|█████████████████████████████████████████████████████████| 7/7 [00:00<00:00, 256.49it/s]

Retried: 7
Now succeeded: 0
Still failed: 7
[{'folder': 'C:\\F1-AI\\data\\raw\\fastf1\\2021\\Russian_Grand_Prix\\Practice_3', 'error_type': 'FileNotFoundError', 'error_message': "[Errno 2] No such file or directory: 'C:\\\\F1-AI\\\\data\\\\raw\\\\fastf1\\\\2021\\\\Russian_Grand_Prix\\\\Practice_3\\\\laps.parquet'"}, {'folder': 'C:\\F1-AI\\data\\raw\\fastf1\\2020\\Styrian_Grand_Prix\\Practice_3', 'error_type': 'FileNotFoundError', 'error_message': "[Errno 2] No such file or directory: 'C:\\\\F1-AI\\\\data\\\\raw\\\\fastf1\\\\2020\\\\Styrian_Grand_Prix\\\\Practice_3\\\\laps.parquet'"}, {'folder': 'C:\\F1-AI\\data\\raw\\fastf1\\2020\\Eifel_Grand_Prix\\Practice_1', 'error_type': 'FileNotFoundError', 'error_message': "[Errno 2] No such file or directory: 'C:\\\\F1-AI\\\\data\\\\raw\\\\fastf1\\\\2020\\\\Eifel_Grand_Prix\\\\Practice_1\\\\laps.parquet'"}, {'folder': 'C:\\F1-AI\\data\\raw\\fastf1\\2020\\Eifel_Grand_Prix\\Practice_2', 'error_type': 'FileNotFoundError', 'error_message': "[Errno 2

In [63]:
for folder_path in [
    r"C:\F1-AI\data\raw\fastf1\2021\Russian_Grand_Prix\Practice_3",
    r"C:\F1-AI\data\raw\fastf1\2018\Italian_Grand_Prix\Race",
]:
    p = Path(folder_path)
    print(p)
    print([f.name for f in p.iterdir()])
    print()

C:\F1-AI\data\raw\fastf1\2021\Russian_Grand_Prix\Practice_3
['car_data', 'download_log.json', 'metadata.json', 'position_data', 'results.parquet', 'weather.parquet']

C:\F1-AI\data\raw\fastf1\2018\Italian_Grand_Prix\Race
['car_data', 'download_log.json', 'metadata.json', 'position_data', 'results.parquet', 'weather.parquet']



### **Known gap: 7 sessions missing laps.parquet**
______________________________________________________________

During the full archive run, 7 out of 857 sessions failed because
`laps.parquet` itself was missing from the raw download folder
(likely a rate limit or timeout during the original 24 hour download
in `01_download_fastf1.ipynb`). `results`, `weather`, `car_data`, and
`position_data` are all present for these sessions, only `laps` is gone.

Since `laps` is our fact table (one row = one driver x one lap), these
7 sessions cannot be reconstructed without a fresh download, and were
excluded from the processed archive.

Affected sessions:
- 2021 Russian GP, Practice 3
- 2020 Styrian GP, Practice 3
- 2020 Eifel GP, Practice 1
- 2020 Eifel GP, Practice 2
- 2019 Japanese GP, Practice 3
- 2018 Italian GP, Race
- 2018 German GP, Practice 1

Final processed coverage: 850 out of 857 sessions (99.2%).

This gap is small, scattered across different seasons rather than
concentrated in one, and therefore not expected to meaningfully affect
any of the four planned ML models.

In [65]:
saved_files = list(PROCESSED_DIR.glob("*.parquet"))

print("Total processed session files on disk:", len(saved_files))

Total processed session files on disk: 850


In [66]:
total_size_mb = sum(f.stat().st_size for f in saved_files) / (1024 * 1024)

print("Total size of all processed session files:", round(total_size_mb, 2), "MB")
print("Number of files:", len(saved_files))

Total size of all processed session files: 97.27 MB
Number of files: 850


In [67]:
all_session_dfs = []

for file_path in tqdm(saved_files, desc="Loading processed sessions"):
    df = pd.read_parquet(file_path)
    all_session_dfs.append(df)

master_df = pd.concat(all_session_dfs, ignore_index=True)

print("Master table shape:", master_df.shape)

Loading processed sessions: 100%|████████████████████████████████████████████████████| 850/850 [00:38<00:00, 21.84it/s]


Master table shape: (469012, 68)


In [68]:
print("Total rows:", len(master_df))
print("Total sessions:", master_df.groupby(["Season", "EventName", "SessionName"]).ngroups)
print("Average rows per session:", round(len(master_df) / master_df.groupby(["Season", "EventName", "SessionName"]).ngroups, 1))

Total rows: 469012
Total sessions: 850
Average rows per session: 551.8


In [71]:
pd.set_option('display.max_columns', None)
master_df.head(10)

,Time,Driver,DriverNumber,LapTime,LapNumber,Stint,PitOutTime,PitInTime,Sector1Time,Sector2Time,Sector3Time,Sector1SessionTime,Sector2SessionTime,Sector3SessionTime,SpeedI1,SpeedI2,SpeedFL,SpeedST,IsPersonalBest,Compound,TyreLife,FreshTyre,Team,LapStartTime,LapStartDate,TrackStatus,Position,Deleted,DeletedReason,FastF1Generated,IsAccurate,BroadcastName,Abbreviation,DriverId,TeamColor,TeamId,FirstName,LastName,FullName,HeadshotUrl,CountryCode,FinalPosition,ClassifiedPosition,GridPosition,Q1,Q2,Q3,ResultTime,Status,Points,Laps,AirTemp,Humidity,Pressure,Rainfall,TrackTemp,WindDirection,WindSpeed,avg_speed,max_speed,avg_throttle,full_throttle_pct,max_rpm,brake_sample_pct,drs_active_pct,Season,EventName,SessionName
0,0 days 00:16:44.279000,RAI,7,NaT,1.0,1.0,0 days 00:14:32.810000,0 days 00:16:43.205000,NaT,0 days 00:00:53.588000,0 days 00:00:52.956000,NaT,0 days 00:15:51.323000,0 days 00:16:44.369000,158.0,198.0,NaN,236.0,False,HYPERSOFT,1.0,True,Ferrari,0 days 00:14:32.810000,2018-11-23 09:00:08.128,1,NaN,False,,False,False,K RAIKKONEN,RAI,raikkonen,DC0000,ferrari,Kimi,Räikkönen,Kimi Räikkönen,,,NaN,,NaN,NaT,NaT,NaT,NaT,,NaN,NaN,29.1,36.9,1016.7,0.0,43.4,108.0,0.4,142.090573,241.0,51.772643,35.120148,12563.0,0.181146,0.000000,2018,Abu_Dhabi_Grand_Prix,Practice_1
1,0 days 00:16:59.665000,ERI,9,NaT,1.0,1.0,0 days 00:14:36.454000,0 days 00:16:58.524000,NaT,0 days 00:01:02.254000,0 days 00:00:57.293000,NaT,0 days 00:16:02.428000,0 days 00:16:59.665000,215.0,168.0,NaN,171.0,False,HYPERSOFT,1.0,True,Sauber,0 days 00:14:36.454000,2018-11-23 09:00:11.772,1,NaN,False,,False,False,M ERICSSON,ERI,ericsson,9B0000,sauber,Marcus,Ericsson,Marcus Ericsson,,,NaN,,NaN,NaT,NaT,NaT,NaT,,NaN,NaN,29.1,36.9,1016.7,0.0,43.4,108.0,0.4,129.984720,257.0,41.821732,12.393888,12133.0,0.179966,0.000000,2018,Abu_Dhabi_Grand_Prix,Practice_1
2,0 days 00:17:00.879000,VAN,2,NaT,1.0,1.0,0 days 00:14:58.922000,0 days 00:16:59.711000,NaT,0 days 00:00:49.023000,0 days 00:00:51.721000,NaT,0 days 00:16:09.158000,0 days 00:17:00.891000,258.0,258.0,NaN,258.0,False,HYPERSOFT,1.0,True,McLaren,0 days 00:14:58.922000,2018-11-23 09:00:34.240,1,NaN,False,,False,False,S VANDOORNE,VAN,vandoorne,FF8700,mclaren,Stoffel,Vandoorne,Stoffel Vandoorne,,,NaN,,NaN,NaT,NaT,NaT,NaT,,NaN,NaN,29.1,36.9,1016.7,0.0,43.4,108.0,0.4,152.958084,260.0,46.211577,0.000000,11950.0,0.191617,0.998004,2018,Abu_Dhabi_Grand_Prix,Practice_1
3,0 days 00:17:07.203000,KUB,40,NaT,1.0,1.0,0 days 00:15:04.381000,NaT,NaT,0 days 00:00:51.612000,0 days 00:00:48.572000,NaT,0 days 00:16:18.631000,0 days 00:17:07.293000,214.0,231.0,199.0,252.0,False,SUPERSOFT,1.0,True,Williams,0 days 00:15:04.381000,2018-11-23 09:00:39.699,1,NaN,False,,False,False,R KUBICA,KUB,nan,FFFFFF,nan,Robert,Kubica,Robert Kubica,,,NaN,,NaN,NaT,NaT,NaT,NaT,,NaN,NaN,29.1,36.9,1016.7,0.0,43.4,108.0,0.4,152.334653,283.0,45.798020,23.564356,11707.0,0.275248,0.000000,2018,Abu_Dhabi_Grand_Prix,Practice_1
4,0 days 00:17:29.430000,SAI,55,NaT,1.0,1.0,0 days 00:15:09.463000,0 days 00:17:27.869000,NaT,0 days 00:00:57.787000,0 days 00:01:01.112000,NaT,0 days 00:16:28.318000,0 days 00:17:29.511000,215.0,219.0,NaN,222.0,False,ULTRASOFT,1.0,True,Renault,0 days 00:15:09.463000,2018-11-23 09:00:44.781,1,NaN,False,,False,False,C SAINZ,SAI,sainz,FFF500,renault,Carlos,Sainz,Carlos Sainz,,,NaN,,NaN,NaT,NaT,NaT,NaT,,NaN,NaN,29.1,36.9,1016.7,0.0,43.4,108.0,0.4,133.170139,276.0,35.619792,14.236111,11751.0,0.227431,5.208333,2018,Abu_Dhabi_Grand_Prix,Practice_1
5,0 days 00:17:19.646000,GRO,8,NaT,1.0,1.0,0 days 00:15:14.664000,0 days 00:17:18.478000,NaT,0 days 00:00:53.154000,0 days 00:00:51.088000,NaT,0 days 00:16:28.558000,0 days 00:17:19.666000,216.0,218.0,NaN,231.0,False,ULTRASOFT,1.0,True,Haas F1 Team,0 days 00:15:14.664000,2018-11-23 09:00:49.982,1,NaN,False,,False,False,R GROSJEAN,GRO,grosjean,828282,haas,Romain,Grosjean,Romain Grosjean,,,NaN,,NaN,NaT,NaT,NaT,NaT,,NaN,NaN,29.1,36.9,1016.7,0.0,43.4,108.0,0.4,149.467836,316.0,36.553606,14.814815,11845.0,0.257310,4.873294,

____
**Master table** `master_df` now exists, and from here it branches into 4 separate paths, one per model, each with its own target label and its own feature engineering on top of this shared base.

_______
This master table itself needs a few checks first, since it's now spanning 2018 to 2025, multiple things could be silently inconsistent across that range that weren't visible when we were testing on one session at a time.

### **Multiple Checks**

In [72]:
print("Season coverage:")
print(master_df["Season"].value_counts().sort_index())

Season coverage:
Season
2018    56548
2019    58945
2020    46500
2021    60293
2022    59565
2023    57489
2024    64039
2025    65633
Name: count, dtype: int64


In [73]:
print("Exact duplicate rows in master table:", master_df.duplicated().sum())

Exact duplicate rows in master table: 0


In [74]:
print("All compound values seen across full archive, by season:")
print(master_df.groupby("Season")["Compound"].value_counts())

All compound values seen across full archive, by season:
Season  Compound    
2018    SOFT            14040
        ULTRASOFT       13752
        SUPERSOFT       13551
        HYPERSOFT        7817
        MEDIUM           5100
        INTERMEDIATE      947
        nan               908
        WET               256
        HARD              117
        None               60
2019    SOFT            29618
        MEDIUM          17143
        HARD            10385
        INTERMEDIATE     1337
        TEST              231
        WET               159
        nan                42
        None               30
2020    SOFT            19073
        MEDIUM          13728
        HARD            10446
        INTERMEDIATE     1197
        WET              1088
        TEST_UNKNOWN      928
        None               23
        UNKNOWN            17
2021    SOFT            22562
        MEDIUM          18702
        HARD            15046
        INTERMEDIATE     2861
        TEST_UNKNOWN  

In [79]:
DRY_COMPOUND_ORDER_2018 = ["HYPERSOFT", "ULTRASOFT", "SUPERSOFT", "SOFT", "MEDIUM", "HARD"]

season_2018 = master_df[master_df["Season"] == 2018]

mapping_rows = []

for event_name, group in season_2018.groupby("EventName"):
    present = [c for c in DRY_COMPOUND_ORDER_2018 if c in group["Compound"].unique()]

    if len(present) == 0:
        continue

    if len(present) == 3:
        rank_map = {present[0]: "SOFT", present[1]: "MEDIUM", present[2]: "HARD"}
    elif len(present) < 3:
        labels = ["SOFT", "MEDIUM", "HARD"][:len(present)]
        rank_map = dict(zip(present, labels))
    else:
        rank_map = {present[0]: "SOFT", present[-1]: "HARD"}
        for mid in present[1:-1]:
            rank_map[mid] = "MEDIUM"

    for raw_compound, category in rank_map.items():
        mapping_rows.append({
            "EventName": event_name,
            "Compound": raw_compound,
            "Compound_category": category
        })

compound_2018_mapping = pd.DataFrame(mapping_rows)

print(compound_2018_mapping.to_string())

Empty DataFrame
Columns: []
Index: []


In [80]:
print(master_df["Season"].dtype)
print(master_df["Season"].unique()[:5])

object
['2018' '2019' '2020' '2021' '2022']


In [81]:
master_df["Season"] = master_df["Season"].astype(int)

print(master_df["Season"].dtype)
print(master_df["Season"].unique())

int64
[2018 2019 2020 2021 2022 2023 2024 2025]


In [82]:
def parse_session_path(session_dir):
    session_name = session_dir.name
    event_name = session_dir.parent.name
    season = int(session_dir.parent.parent.name)
    return season, event_name, session_name

In [83]:
DRY_COMPOUND_ORDER_2018 = ["HYPERSOFT", "ULTRASOFT", "SUPERSOFT", "SOFT", "MEDIUM", "HARD"]

season_2018 = master_df[master_df["Season"] == 2018]

print("Rows in season_2018:", len(season_2018))

Rows in season_2018: 56548


In [84]:
mapping_rows = []

for event_name, group in season_2018.groupby("EventName"):
    present = [c for c in DRY_COMPOUND_ORDER_2018 if c in group["Compound"].unique()]

    if len(present) == 0:
        continue

    if len(present) == 3:
        rank_map = {present[0]: "SOFT", present[1]: "MEDIUM", present[2]: "HARD"}
    elif len(present) < 3:
        labels = ["SOFT", "MEDIUM", "HARD"][:len(present)]
        rank_map = dict(zip(present, labels))
    else:
        rank_map = {present[0]: "SOFT", present[-1]: "HARD"}
        for mid in present[1:-1]:
            rank_map[mid] = "MEDIUM"

    for raw_compound, category in rank_map.items():
        mapping_rows.append({
            "EventName": event_name,
            "Compound": raw_compound,
            "Compound_category": category
        })

compound_2018_mapping = pd.DataFrame(mapping_rows)

print(compound_2018_mapping.to_string())

                   EventName   Compound Compound_category
0       Abu_Dhabi_Grand_Prix  HYPERSOFT              SOFT
1       Abu_Dhabi_Grand_Prix  ULTRASOFT            MEDIUM
2       Abu_Dhabi_Grand_Prix  SUPERSOFT              HARD
3      Australian_Grand_Prix  ULTRASOFT              SOFT
4      Australian_Grand_Prix  SUPERSOFT            MEDIUM
5      Australian_Grand_Prix       SOFT              HARD
6        Austrian_Grand_Prix  ULTRASOFT              SOFT
7        Austrian_Grand_Prix  SUPERSOFT            MEDIUM
8        Austrian_Grand_Prix       SOFT              HARD
9      Azerbaijan_Grand_Prix  ULTRASOFT              SOFT
10     Azerbaijan_Grand_Prix  SUPERSOFT            MEDIUM
11     Azerbaijan_Grand_Prix       SOFT              HARD
12        Bahrain_Grand_Prix  SUPERSOFT              SOFT
13        Bahrain_Grand_Prix       SOFT            MEDIUM
14        Bahrain_Grand_Prix     MEDIUM              HARD
15        Belgian_Grand_Prix  SUPERSOFT              SOFT
16        Belg

In [85]:
season_2018_mapped = season_2018.merge(
    compound_2018_mapping,
    on=["EventName", "Compound"],
    how="left"
)

print("Rows before:", len(season_2018))
print("Rows after:", len(season_2018_mapped))
print("Unmapped 2018 rows (wet weather compounds, expected):")
print(season_2018_mapped[season_2018_mapped["Compound_category"].isna()]["Compound"].value_counts())

Rows before: 56548
Rows after: 56548
Unmapped 2018 rows (wet weather compounds, expected):
Compound
INTERMEDIATE    947
nan             908
WET             256
None             60
Name: count, dtype: int64


In [86]:
JUNK_LABELS = ["nan", "None", "", "TEST", "TEST_UNKNOWN", "UNKNOWN"]

def build_compound_category(row):
    if row["Season"] == 2018:
        return row["Compound_category"] if pd.notna(row["Compound_category"]) else row["Compound"]
    return row["Compound"]

season_2018_final = season_2018_mapped.copy()
season_2018_final["Compound_category"] = season_2018_final.apply(build_compound_category, axis=1)

other_seasons = master_df[master_df["Season"] != 2018].copy()
other_seasons["Compound_category"] = other_seasons["Compound"]

master_with_compound = pd.concat([season_2018_final, other_seasons], ignore_index=True)

master_with_compound["Compound_category"] = master_with_compound["Compound_category"].astype(str).replace(JUNK_LABELS, "UNKNOWN")

print("Final Compound_category distribution:")
print(master_with_compound["Compound_category"].value_counts())
print()
print("Total rows check:", len(master_with_compound) == len(master_df))

Final Compound_category distribution:
Compound_category
SOFT            181434
MEDIUM          147961
HARD            109712
INTERMEDIATE     19349
UNKNOWN           6611
WET               3945
Name: count, dtype: int64

Total rows check: True


**SANITY CHECKS**
___

In [87]:
print("Final shape:", master_with_compound.shape)
print()
print("Dtype summary:")
print(master_with_compound.dtypes.value_counts())

Final shape: (469012, 69)

Dtype summary:
float64            26
object             23
timedelta64[ns]    15
bool                3
datetime64[ns]      1
int64               1
Name: count, dtype: int64


In [88]:
critical_cols = ["Driver", "DriverNumber", "LapNumber", "LapTime", "Compound_category", "TyreLife", "Season", "EventName", "SessionName"]

print("Nulls in critical columns:")
print(master_with_compound[critical_cols].isna().sum())

Nulls in critical columns:
Driver                   0
DriverNumber             0
LapNumber                0
LapTime              62092
Compound_category        0
TyreLife              2586
Season                   0
EventName                0
SessionName              0
dtype: int64


In [89]:
FINAL_OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "fastf1_ml_base.parquet"

master_with_compound.to_parquet(FINAL_OUTPUT_PATH, index=False)

print("Saved to:", FINAL_OUTPUT_PATH)
print("File size:", round(FINAL_OUTPUT_PATH.stat().st_size / (1024*1024), 2), "MB")

Saved to: C:\F1-AI\data\processed\fastf1_ml_base.parquet
File size: 48.01 MB


### **Conclusion — Notebook 05**

#### Objective recap

This notebook implemented the integration strategy decided in
`03_prepare_fastf1.ipynb`, building the FastF1 machine-learning base
table at the grain:

> **One row = One Driver x One Lap**

##### What was built

- Verified the full merge pipeline on one representative session first
  (2018 Abu Dhabi GP, Practice 1) before scaling, checking row counts
  after every single join.
- `laps` merged with `results` (session-level driver/team metadata),
  clean 1:1 key match on DriverNumber, zero row loss.
- `weather` time-aligned to each lap's start time using merge_asof
  (backward direction), deliberately using LapStartTime rather than
  lap end time, to respect information-time correctness. No feature
  should reflect weather conditions that hadn't occurred yet when the
  lap began.
- `car_data` (high-frequency per-driver telemetry) aggregated from
  raw samples down to 7 lap-level summary features: avg_speed,
  max_speed, avg_throttle, full_throttle_pct, max_rpm,
  brake_sample_pct, drs_active_pct.
- `position_data` deliberately left unused. Status showed zero
  variance (always OnTrack) in inspection, and X/Y/Z track coordinates
  don't translate into a useful lap-level feature for our four
  planned targets without significant extra engineering. Revisit only
  if a future model (e.g. overtake detection) specifically needs it.

##### Bugs caught and fixed during this notebook

- Throttle sensor value 104.0 is a known FastF1 error code for a
  stationary car, was silently inflating avg_throttle on red-flag and
  pit-affected laps until caught and excluded.
- Season column was being parsed as string, not int, from folder
  names, silently broke a season-based filter with zero error
  (empty result, not a crash). Fixed at the source and patched
  in-memory.
- Compound naming is inconsistent across eras: 2018 used the old
  6-tier naming (HYPERSOFT/ULTRASOFT/SUPERSOFT/SOFT/MEDIUM/HARD),
  2019 onward uses relative per-weekend naming (SOFT/MEDIUM/HARD only).
  Built an event-by-event remapping for 2018 based on actual compound
  hardness ranking used at each race, so Compound_category is now
  consistent and usable across the full 2018-2025 span. Original
  Compound column kept unchanged for traceability.
- Junk/placeholder compound labels (TEST, TEST_UNKNOWN, UNKNOWN, nan,
  None, empty string) collapsed into a single UNKNOWN category across
  all seasons.

##### Known gaps (accepted, documented, not blocking)

- 7 out of 857 sessions excluded entirely, laps.parquet was missing
  from the raw archive for these (likely a rate-limit/timeout during
  the original 24-hour download in `01`). Scattered across different
  seasons, not concentrated, not expected to bias any of the 4 models.
- 8 sessions have weather features as null (weather.parquet was
  missing from raw archive), lap/results/telemetry data intact for
  these.

##### Final output

- 850 of 857 sessions successfully processed (99.2% coverage).
- Final combined table: 469,012 rows x 69 columns.
- Saved to: `data/processed/fastf1_ml_base.parquet` (48.01 MB).
- This is the shared base table all four downstream models
  (lap time, tyre degradation, pit stop strategy, race outcome) will
  build their specific features and targets from.

##### What this notebook did NOT do (by design)

- No target/label creation for any specific model.
- No feature engineering beyond the shared telemetry aggregation.
- No EDA on the combined table itself.

##### Next step

EDA on `fastf1_ml_base.parquet`, understanding distributions, per-era
compound behavior now that it's normalized, degradation patterns,
pit stop frequency patterns, before deciding exact features and
targets for each of the four models.